# Experiment: episode length x inference context (seed 42)

**Requires a GPU.** This is the run that produced `data/curves_seed42.csv`.
It is included as provenance; you do not need to run it to check the results -
`02_analysis.ipynb` recomputes everything from the committed CSV, on CPU.

**What it does.** For each of 24 binary OpenML tasks (suite 457), fine-tunes
TabPFN 3.5 with episodes of a fixed length, checkpoints at 25 / 50 / 200 gradient
steps, and evaluates each checkpoint across a sweep of inference context lengths.
The un-adapted model is measured at every context as the reference.

**Protocol.** Fold 0 is sealed and never read. The outer training split is cut
80/20 into fit and report; the fit part is cut 70/15/15 into train, Vscore and
Vselect. Metrics are computed on the report split only.

*Set the Kaggle secret `TABPFN_TOKEN` and select a GPU accelerator before
running; neither follows a copied notebook.*

---


## Context against fine-tuning: is one a substitute for the other?

**The question.** Earlier fine-tuning studies all measured at **1024 context
rows**, and 1024 is already close to saturation for most of these datasets. So a
fine-tuning gain measured there is being asked to add something on top of an
almost-converged model. This notebook measures the same quantity across the whole
context axis, from 32 rows to the full training split, on 24 binary OpenML tasks.

**What is varied.** Three axes, crossed:

* the **inference context**: 32, 64, 128, 256, 512, 1024, 2048, and the full
  training split;
* the **fine-tuning budget**: 0 (un-adapted), 25, 50 and 200 gradient steps,
  checkpointed from a single training run per dataset;
* the **episode length** (phase C): 128-row episodes against 640-row episodes,
  on the largest datasets.

**What is held fixed.** The splits, the seed, the evaluation rows, the learning
rate, and the number of gradient steps between compared arms. Fold 0 is sealed
and never read. Metrics are computed on a report split that no decision touches.

**The protocol is frozen before any result is looked at**, and the predictions
are printed by the notebook itself in section 7, so they are timestamped by the
run rather than written afterwards.

In [ ]:

import subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','-q','tabpfn==9.0.0',
                'openml>=0.15,<0.17','scikit-learn>=1.5,<1.8','scipy','matplotlib','nbformat>=5'],check=True)
print('Dependencies installed. Restart the kernel if Kaggle asks, then run all.')

In [ ]:

import os
os.environ['TABPFN_NO_BROWSER']='1'
if not os.environ.get('TABPFN_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('TABPFN_TOKEN')
    except Exception as exc:
        raise RuntimeError('Create and enable the Kaggle secret TABPFN_TOKEN (Prior Labs API key).') from exc
    if not token: raise RuntimeError('The TABPFN_TOKEN secret is empty.')
    os.environ['TABPFN_TOKEN']=token.strip(); del token
print('Credentials loaded in memory, key never printed.')

## 1. Configuration

Everything is decided here, **before** any result is seen. The `RUN_HASH` makes
it impossible to mix two configurations in one output folder.

In [ ]:

from pathlib import Path

SEED = 42
SUITE_ID = 457
ALL_TASK_IDS = [363621,363626,363619,363623,363682,363684,
                363613,363618,363624,363627,363629,363632,
                363671,363674,363676,363679,363681,363689,
                363691,363694,363696,363700,363706,363712]

# Optional split into sessions. 1 shard = everything in one run.
N_SHARDS, SHARD_ID = 1, 0

# ---- axis 1: the inference context ----------------------------------------
# We go down to 32 rows: that is where fine-tuning would win if it wins anywhere.
# None = the target's whole training split.
CONTEXT_SIZES     = (32,64,128,256,512,1024,2048,None)
REF_CONTEXT       = 1024      # the operating point of the three earlier runs
SMALL_CONTEXT_MAX = 256       # below this, drawing the context is noisy
REPEATS_SMALL     = 3         # so we repeat it and average the metrics

# ---- axis 2: the fine-tuning budget ---------------------------------------
# One training run per dataset, checkpointed: the three budgets cost only
# the longest of them. 50 is the setting used in earlier runs.
FT_MILESTONES = (25,50,200)
FT_CONTEXT, FT_QUERY = 102, 26            # 128-row episode, query share 0.2
LEARNING_RATE, WEIGHT_DECAY, GRAD_CLIP = 1e-5, 0.01, 1.0

# ---- axis 3 (phase C): the length of a training episode -------------------
# Answers the objection "your fine-tuning is specialised for 100 rows".
RUN_LONG_EPISODE_PHASE = True
LONG_FT_CONTEXT, LONG_FT_QUERY = 512, 128  # 640-row episode
LONG_FT_MILESTONES = FT_MILESTONES         # SAME budgets, or nothing is comparable
LONG_PHASE_TASKS = 4                       # the 4 largest training splits

# ---- evaluation ----------------------------------------------------------
REPORT_EVAL_CAP = 1024        # rows of the sealed report split, identical for every arm
PRED_BATCHES    = (64,16,4,1)

# ---- time budget ---------------------------------------------------------
# Each phase checks elapsed time BEFORE starting a new dataset.
# An overlong run stops cleanly and writes what it has: nothing is ever lost.
TIME_BUDGET_HOURS = 8.0

# OpenML returns 503 regularly; retry rather than lose the run.
OPENML_RETRIES, OPENML_BACKOFF = 6, 10

SKIP_ON_ERROR = True
MIN_TARGETS = 12
MAX_DATASET_ROWS, MAX_FEATURES = 50_000, 110
MIN_FREE_GIB = 3.0
LOCAL_TRAIN_CAP = REF_CONTEXT   # name expected by the reused split code

RUN_TAG = 'contexte_x_finetuning_v1'
ON_KAGGLE = Path('/kaggle/working').exists()
ROOT    = (Path('/kaggle/working') if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG   # CSVs, kept
SCRATCH = (Path('/kaggle/temp')   if ON_KAGGLE else Path('/mnt/data'))/RUN_TAG    # weights, ephemeral
ROOT.mkdir(parents=True,exist_ok=True); SCRATCH.mkdir(parents=True,exist_ok=True)

SHARD_IDS = ALL_TASK_IDS[SHARD_ID::N_SHARDS]
assert len(ALL_TASK_IDS)==len(set(ALL_TASK_IDS))==24
assert 0<=SHARD_ID<N_SHARDS and SHARD_IDS
assert REF_CONTEXT in CONTEXT_SIZES and None in CONTEXT_SIZES
assert list(FT_MILESTONES)==sorted(set(FT_MILESTONES)) and min(FT_MILESTONES)>=1
assert list(LONG_FT_MILESTONES)==sorted(set(LONG_FT_MILESTONES))
assert all(c is None or c>=8 for c in CONTEXT_SIZES)

print('Results:',ROOT,'| temporary weights:',SCRATCH)
print('Contexts   :',CONTEXT_SIZES,'(None = the whole training split)')
print('FT budgets:',(0,)+tuple(FT_MILESTONES),'steps | episode of',FT_CONTEXT+FT_QUERY,'rows')
print('Phase C    :',('ON, episodes of %d rows over %d datasets'%(LONG_FT_CONTEXT+LONG_FT_QUERY,LONG_PHASE_TASKS))
      if RUN_LONG_EPISODE_PHASE else 'OFF')
print(f'Time budget: {TIME_BUDGET_HOURS} h')
print('Expected order of magnitude on a T4: phase A ~0.6 h, phase B ~4.5 h, phase C ~1.5 h.')
print('Phase C runs last and stops itself: a partial result there stays '
      'readable, since the comparison is paired dataset by dataset.')

## 2. Imports and utilities

The fine-tuning machinery is included. Internal APIs are pinned to
`tabpfn==9.0.0`: any change in that version invalidates the run, and the
assertions below say so rather than failing silently.

In [ ]:

import re, gc, json, time, random, shutil, hashlib, inspect, traceback, importlib.metadata
import numpy as np, pandas as pd, torch, openml
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, log_loss, average_precision_score
from scipy.stats import spearmanr
from IPython.display import display

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from tabpfn.base import get_embeddings as tabpfn_get_embeddings
from tabpfn.model_loading import save_tabpfn_model
from tabpfn.finetuning.finetuned_classifier import FinetunedTabPFNClassifier
from tabpfn.finetuning.finetuned_base import _parameters_unused_by_task
from tabpfn.finetuning.data_util import get_preprocessed_dataset_chunks, meta_dataset_collator
from tabpfn.finetuning._torch_compat import GradScaler, autocast, sdpa_kernel_context
from tabpfn.architectures.interface import PerformanceOptions

assert importlib.metadata.version('tabpfn')=='9.0.0', 'tabpfn != 9.0.0: restart the kernel after pip.'
assert torch.cuda.is_available(), 'Enable the GPU (Settings -> Accelerator -> GPU).'
assert hasattr(ModelVersion,'V3_5')
assert 'shuffle' in inspect.signature(get_preprocessed_dataset_chunks).parameters
DEVICE='cuda:0'; GIB=2**30
print('GPU :',torch.cuda.get_device_name(0),'/',round(torch.cuda.get_device_properties(0).total_memory/GIB,2),'Gio')

def clean_gpu():
    gc.collect(); torch.cuda.empty_cache()

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def sha(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,default=str).encode()).hexdigest()

def sha_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

def write_json(path,obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp')
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
    tmp.replace(p)

def write_csv(path,frame):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_name(p.name+'.tmp'); frame.to_csv(tmp,index=False); tmp.replace(p)

def check_disk(label,where=None,extra=0.0):
    free=shutil.disk_usage(where or SCRATCH).free/GIB
    if free < MIN_FREE_GIB+extra:
        raise RuntimeError(f'{label}: {free:.1f} GiB free. Export the CSVs, then re-run a smaller shard.')

def choose_indices(y,n,seed):
    # Deterministic stratified subsample, at least 2 rows per class.
    y=np.asarray(y); n=min(int(n),len(y))
    classes,counts=np.unique(y,return_counts=True)
    if len(classes)!=2 or min(counts)<2: raise ValueError('Two classes with >= 2 rows required')
    if n==len(y): return np.arange(len(y))
    if n<4: raise ValueError('Sample too small')
    k1=max(2,min(int(counts[1]),int(round(n*counts[1]/len(y))))); k0=n-k1
    if k0>counts[0]: k0=int(counts[0]); k1=n-k0
    if k0<2: k0=2; k1=n-k0
    if k1>counts[1] or k0>counts[0] or min(k0,k1)<2: raise ValueError('Stratification impossible')
    rng=np.random.default_rng(int(seed))
    a=rng.choice(np.flatnonzero(y==classes[0]),k0,replace=False)
    b=rng.choice(np.flatnonzero(y==classes[1]),k1,replace=False)
    return np.sort(np.r_[a,b])

def scores(y,p):
    p=np.asarray(p,dtype=np.float64).reshape(-1)
    if not np.isfinite(p).all() or (p<0).any() or (p>1).any(): raise ValueError('Invalid probabilities')
    if len(np.unique(y))!=2: raise ValueError('Metric undefined on a single-class split')
    return dict(auc=float(roc_auc_score(y,p)),
                logloss=float(log_loss(y,np.clip(p,1e-7,1-1e-7),labels=[0,1])),
                ap=float(average_precision_score(y,p)))

def is_oom(e):
    return isinstance(e,torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()

FAILURES=[]
def record_failure(phase,tid,name,exc,**extra):
    # A failure is dropped from the run, never hidden: it lands in failures.csv and on screen.
    row=dict(phase=phase,task_id=tid,name=name,error=type(exc).__name__,
             message=str(exc)[:400],**extra)
    FAILURES.append(row)
    p=ROOT/'failures.csv'
    tmp=p.with_name(p.name+'.tmp'); pd.DataFrame(FAILURES).to_csv(tmp,index=False); tmp.replace(p)
    print(f'  !! DROPPED [{phase}] {name} ({tid}) - {type(exc).__name__}: {str(exc)[:160]}',flush=True)
    if not SKIP_ON_ERROR: raise exc
    return row

def openml_retry(fn,*a,**kw):
    # OpenML returns 503 regularly, or cuts out mid-download. We retry with
    # growing backoff rather than lose a run of several hours.
    last=None
    for k in range(OPENML_RETRIES):
        try: return fn(*a,**kw)
        except Exception as exc:
            last=exc
            if k==OPENML_RETRIES-1: break
            wait=min(OPENML_BACKOFF*(2**k),120)
            print(f'    OpenML unavailable ({type(exc).__name__}) - retrying '
                  f'dans {wait}s [{k+1}/{OPENML_RETRIES-1}]',flush=True)
            time.sleep(wait)
    raise last

set_seed(SEED)
print('Utilities ready.')

## 3. Metadata and splits

Outer fold 0 is sealed; the outer training split is cut 80% fit / 20% report,
then the fit part is cut 70% train / 15% Vscore / 15% Vselect. Vscore and Vselect
are not used here — no hyperparameter is chosen — but the split is built anyway
so that the protocol matches the other notebooks exactly.

In [ ]:

openml.config.set_root_cache_directory(str(SCRATCH/'openml_cache'))
check_disk('OpenML metadata')
# Suite membership: a useful check, but not at the cost of the run.
# The 24 task ids are frozen and were already validated in earlier runs.
try:
    suite=openml_retry(openml.study.get_suite,SUITE_ID)
    allowed=set(map(int,suite.tasks))
    assert set(ALL_TASK_IDS)<=allowed
    print('Suite 457 verified.')
except Exception as exc:
    print(f'Suite 457 not verifiable ({type(exc).__name__}) - continuing with the '
          '24 frozen task ids.',flush=True)

def clean_description(s):
    s=re.sub(r'<[^>]+>',' ',str(s or ''))
    s=re.sub(r'https?://\S+',' ',s)
    return re.sub(r'\s+',' ',s).strip()[:2500]

META={}; TASKS={}; TEXT={}; seen=set()
for tid in list(ALL_TASK_IDS):
    try:
        t=openml_retry(openml.tasks.get_task,tid,download_data=False,download_qualities=False)
        ds=openml_retry(t.get_dataset,download_data=False)
        did=int(t.dataset_id)
        if did in seen: raise ValueError(f'{tid}: duplicate OpenML dataset_id')
        txt=f'{ds.name}. Prediction target {t.target_name}. '+clean_description(ds.description)
        if len(txt)<20: raise ValueError(f'{tid}: description too short for the text baseline')
        seen.add(did)
        META[tid]={'task_id':tid,'dataset_id':did,'name':str(ds.name),'target':str(t.target_name),
                   'description':txt,'url':f'https://www.openml.org/t/{tid}'}
        TASKS[tid]=t; TEXT[tid]=txt
        print(tid,META[tid]['name'])
    except Exception as exc:
        record_failure('metadata',tid,f'task_{tid}',exc)
ALL_TASK_IDS=[t for t in ALL_TASK_IDS if t in TASKS]
if len(ALL_TASK_IDS)<MIN_TARGETS:
    raise RuntimeError(f'only {len(ALL_TASK_IDS)} tasks: see failures.csv.')
print(f'{len(ALL_TASK_IDS)}/24 OpenML tasks available.')
write_json(ROOT/'catalog_meta.json',META)

In [ ]:

def encode_task(frames):
    # frames[0] = train: the only split the preprocessing is fitted on.
    frames=[f.copy().reset_index(drop=True) for f in frames]
    good=[c for c in frames[0].columns
          if frames[0][c].notna().sum()>0 and frames[0][c].nunique(dropna=True)>1]
    if not good: raise ValueError('No usable feature')
    nums=[c for c in good if pd.api.types.is_numeric_dtype(frames[0][c])]
    cats=[c for c in good if c not in nums]
    parts=[]
    if nums: parts.append(('numeric',SimpleImputer(strategy='median'),nums))
    if cats: parts.append(('categorical',OrdinalEncoder(handle_unknown='use_encoded_value',
                            unknown_value=-1,encoded_missing_value=-1,dtype=np.float32),cats))
    prep=ColumnTransformer(parts,remainder='drop',sparse_threshold=0)
    clean=[]
    for f in frames:
        f=f[good].copy()
        for c in nums: f[c]=pd.to_numeric(f[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
        for c in cats: f[c]=f[c].astype('string').fillna('__MISSING__').astype(str)
        clean.append(f)
    out=[]
    for j,f in enumerate(clean):
        x=prep.fit_transform(f) if j==0 else prep.transform(f)
        x=np.asarray(x,dtype=np.float32)
        x=np.clip(np.nan_to_num(x,nan=0,posinf=1e4,neginf=-1e4),-1e4,1e4)
        if not np.isfinite(x).all(): raise ValueError('Non-finite X values')
        out.append(np.ascontiguousarray(x))
    return out,{'n_features':len(good),'numeric':len(nums),'categorical':len(cats),
                'feature_name_sha256':hashlib.sha256(json.dumps([str(c) for c in good]).encode()).hexdigest()}

def get_bundle(tid):
    task=TASKS[tid]; ds=openml_retry(task.get_dataset)
    X,y,_,_=openml_retry(ds.get_data,target=task.target_name)
    if not isinstance(X,pd.DataFrame) or isinstance(y,pd.DataFrame):
        raise TypeError('Unexpected OpenML X/y format')
    outer,sealed=task.get_train_test_split_indices(fold=0,repeat=0,sample=0)
    outer=np.asarray(outer); sealed=np.asarray(sealed)
    assert np.intersect1d(outer,sealed).size==0 and len(outer)+len(sealed)==len(X)
    label=LabelEncoder().fit(pd.Series(y).iloc[outer].astype(str))
    if len(label.classes_)!=2: raise ValueError(f'{tid}: task is not binary')
    y_outer=label.transform(pd.Series(y).iloc[outer].astype(str))
    if min(np.bincount(y_outer))<20: raise ValueError(f'{tid}: minority class too rare')
    i_fit,i_report=train_test_split(outer,test_size=0.20,stratify=y_outer,random_state=SEED)
    y_fit=label.transform(pd.Series(y).iloc[i_fit].astype(str))
    i_train,i_val=train_test_split(i_fit,test_size=0.30,stratify=y_fit,random_state=SEED+1)
    y_val_tmp=label.transform(pd.Series(y).iloc[i_val].astype(str))
    i_score,i_select=train_test_split(i_val,test_size=0.50,stratify=y_val_tmp,random_state=SEED+2)
    for a,b in [(i_train,i_score),(i_train,i_select),(i_train,i_report),
                (i_score,i_select),(i_score,i_report),(i_select,i_report)]:
        assert np.intersect1d(a,b).size==0, f'{tid}: leakage between splits'
    enc=lambda idx: label.transform(pd.Series(y).iloc[idx].astype(str)).astype(np.int64)
    ytr,ysc,ysl,yrp=enc(i_train),enc(i_score),enc(i_select),enc(i_report)
    for nm,z in [('train',ytr),('Vscore',ysc),('Vselect',ysl),('report',yrp)]:
        if min(np.bincount(z))<4: raise ValueError(f'{tid}: class too rare in {nm}')
    (Xtr,Xsc,Xsl,Xrp),feat=encode_task([X.iloc[i_train],X.iloc[i_score],X.iloc[i_select],X.iloc[i_report]])
    if len(X)>MAX_DATASET_ROWS or Xtr.shape[1]>MAX_FEATURES:
        raise ValueError(f'{tid}: outside the caps ({len(X)} rows, {Xtr.shape[1]} features)')
    ctx=choose_indices(ytr,min(LOCAL_TRAIN_CAP,len(ytr)),SEED+tid)
    b=dict(Xtrain=Xtr,ytrain=ytr,Xscore=Xsc,yscore=ysc,Xselect=Xsl,yselect=ysl,
           Xreport=Xrp,yreport=yrp,Xcontext=Xtr[ctx],ycontext=ytr[ctx],
           context_idx=ctx,cat_indices=[],
           meta={**META[tid],'n_train':len(ytr),'n_score':len(ysc),'n_select':len(ysl),
                 'n_report':len(yrp),'n_context':len(ctx),'features':feat,
                 'sealed_outer_test_rows':len(sealed),
                 'split_sha256':hashlib.sha256(np.sort(i_train).astype('<i8').tobytes()).hexdigest()})
    return b

def bundle_cache_paths(tid):
    # A previous run attached as an input saves depending on OpenML again.
    cands=[ROOT/'bundles'/f'{tid}.npz']
    inp=Path('/kaggle/input')
    if inp.exists(): cands+=sorted(inp.glob(f'**/bundles/{tid}.npz'))
    return cands

def load_bundle(tid):
    for p in bundle_cache_paths(tid):
        if not p.exists(): continue
        try:
            with np.load(p,allow_pickle=False) as a:
                b={k:a[k] for k in a.files if k!='meta_json'}
                b['meta']=json.loads(str(a['meta_json'].item()))
            b['cat_indices']=[]
            return b
        except Exception as exc:
            print(f'  unreadable cache {p} ({type(exc).__name__})',flush=True)
    return None

def save_bundle(tid,b):
    d=ROOT/'bundles'; d.mkdir(parents=True,exist_ok=True)
    p=d/f'{tid}.npz'; tmp=p.with_name(p.name+'.tmp')
    arrays={k:v for k,v in b.items() if isinstance(v,np.ndarray)}
    with tmp.open('wb') as f:
        np.savez_compressed(f,meta_json=np.array(json.dumps(b['meta'],default=str)),**arrays)
    tmp.replace(p)

BUNDLES={}; rows=[]; from_cache=0
for tid in ALL_TASK_IDS:
    try:
        cached=load_bundle(tid)
        if cached is not None:
            BUNDLES[tid]=cached; from_cache+=1
        else:
            check_disk(f'loading {tid}')
            BUNDLES[tid]=get_bundle(tid); save_bundle(tid,BUNDLES[tid])
        m=BUNDLES[tid]['meta']
        rows.append({k:m[k] for k in
            ['task_id','name','n_train','n_score','n_select','n_report','n_context']}
            |{'n_features':m['features']['n_features']})
        print(f"{m['name'][:42]:<42} | train {m['n_train']:>5} Vscore {m['n_score']:>4} "
              f"Vselect {m['n_select']:>4} report {m['n_report']:>5} | context {m['n_context']:>4} "
              f"| variables {m['features']['n_features']:>3}")
    except Exception as exc:
        record_failure('bundle',tid,META[tid]['name'],exc)
        clean_gpu()

print(f'{from_cache}/{len(ALL_TASK_IDS)} datasets re-read from a local or attached cache.')
ACTIVE_IDS=[t for t in ALL_TASK_IDS if t in BUNDLES]
if len(ACTIVE_IDS)<MIN_TARGETS:
    raise RuntimeError(f'Only {len(ACTIVE_IDS)} datasets loaded: see failures.csv.')
PARTITIONS=pd.DataFrame(rows); write_csv(ROOT/'partitions.csv',PARTITIONS); display(PARTITIONS)
write_json(ROOT/'bundles_meta.json',{str(t):BUNDLES[t]['meta'] for t in ACTIVE_IDS})
print(f'\n{len(ACTIVE_IDS)}/24 datasets active - each one serves as target and as donor.')

## 4. The episodic engine

Episode length is a parameter, since it is the axis of phase C. No LoRA, no
projection, no parameter subsetting: the whole model is fine-tuned.

The numerical fallback matters on this hardware. A T4 is `sm_75` and has no
bf16, so autocast falls back to fp16 and the backward pass overflows on wide
tables. The loss scale is reduced over eight steps, then the pass is redone in
fp32, then the episode is redrawn. **The `used_amp` flag is load-bearing**: in
fp32 the gradients are already at scale 1, so the caller must use
`optimizer.step()` and never `scaler.step()`, which would divide them by 65536.

In [ ]:

def stratified_pick(y,n,seed):
    y=np.asarray(y); assert len(np.unique(y))==2
    n=min(int(n),len(y))
    if n==len(y): return np.arange(len(y))
    idx,_=train_test_split(np.arange(len(y)),train_size=n,stratify=y,random_state=int(seed))
    if len(np.unique(y[idx]))<2: raise ValueError('Single-class episode')
    return np.sort(idx)

def ft_episode(b,seed,n_context,n_query):
    # Context and query rows are disjoint, both drawn from the training split.
    for k in range(30):
        s=int(seed)+k*100003
        iq=stratified_pick(b['ytrain'],n_query,s)
        pool=np.flatnonzero(~np.isin(np.arange(len(b['ytrain'])),iq))
        if len(pool)<n_context or len(np.unique(b['ytrain'][pool]))<2: continue
        ic=pool[stratified_pick(b['ytrain'][pool],n_context,s+1)]
        assert not (set(ic.tolist()) & set(iq.tolist()))
        return (b['Xtrain'][ic],b['ytrain'][ic],b['Xtrain'][iq],b['ytrain'][iq],s),ic,iq
    raise RuntimeError('Cannot build a disjoint episode')

class EpisodeEngine:
    def __init__(self,checkpoint,cat_indices,seed):
        set_seed(seed)
        kw=dict(device=DEVICE,n_estimators=1,random_state=int(seed),fit_mode='batched',
                differentiable_input=False,memory_saving_mode=True,
                categorical_features_indices=cat_indices,
                inference_config={'ENABLE_GPU_PREPROCESSING':False,'TRANSFORM_TEXT':False,
                                  'TRANSFORM_DATES':False})
        self.est=(TabPFNClassifier.create_default_for_version(ModelVersion.V3_5,**kw)
                  if checkpoint is None else TabPFNClassifier(model_path=str(checkpoint),**kw))
        self.est._initialize_model_variables(); self.est.model_.to(DEVICE)
        for p in _parameters_unused_by_task(self.est.model_,'multiclass'): p.requires_grad=False
        self.wrapper=FinetunedTabPFNClassifier(device=DEVICE,model_version=ModelVersion.V3_5)
        self.wrapper.finetuned_estimator_=self.est
        self.wrapper._ddp_module_=None; self.wrapper._local_n_estimators_=1
        self.performance=PerformanceOptions(force_recompute_layer=True,use_chunkwise_inference=False)
    def params(self):
        return {n:p for n,p in self.est.model_.named_parameters() if p.requires_grad}
    def batch(self,episode):
        xc,yc,xq,yq,seed=episode; k=len(yc)
        assert set(np.unique(yc))=={0,1}
        def fixed_split(X,y,stratify=None): return X[:k],X[k:],y[:k],y[k:]
        ds=get_preprocessed_dataset_chunks(calling_instance=self.est,
            X_raw=np.ascontiguousarray(np.concatenate([xc,xq]),dtype=np.float32),
            y_raw=np.concatenate([yc,yq]),split_fn=fixed_split,max_data_size=None,
            model_type='classifier',equal_split_size=False,data_shuffle_seed=seed,
            preprocessing_random_state=seed,shuffle=False)
        assert len(ds)==1
        return meta_dataset_collator([ds[0]])
    def loss(self,batch):
        self.est.fit_from_preprocessed(batch.X_context,batch.y_context,batch.cat_indices,
                                       batch.configs,performance_options=self.performance)
        self.est.model_.train()
        return self.wrapper._forward_with_loss(batch)
    def save(self,path):
        p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
        tmp=p.with_name(p.name+'.tmp'); save_tabpfn_model(self.est,tmp); tmp.replace(p)
    def close(self):
        del self.wrapper; del self.est; clean_gpu()

AMP_EVENTS=[]   # log of numerical fallbacks: a measurement in its own right

def _grads_finite(engine):
    return all(p.grad is None or bool(torch.isfinite(p.grad).all())
               for p in engine.params().values())

def backward_checked(engine,batch,optimizer,scaler,tag='',step=None):
    # Returns (loss, used_amp). used_amp=False means gradients are already at scale 1:
    # the caller must use optimizer.step(), never scaler.step().
    scale0=scaler.get_scale()
    for k in range(8):                       # 65536 -> 4, quartering each time
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=True),sdpa_kernel_context(): loss=engine.loss(batch)
        if not torch.isfinite(loss): break   # the problem is in the forward pass, not the backward
        with sdpa_kernel_context(): scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if _grads_finite(engine):
            if k: AMP_EVENTS.append(dict(tag=tag,step=step,mode='amp_rescaled',
                                         reductions=k,scale=scaler.get_scale()))
            return float(loss.detach()),True
        scaler.update(new_scale=max(1.0,scaler.get_scale()/4))
    # fp32 fallback: the fp16 overflow disappears and the step stays exact.
    optimizer.zero_grad(set_to_none=True)
    with autocast(enabled=False),sdpa_kernel_context(): loss=engine.loss(batch)
    if torch.isfinite(loss):
        with sdpa_kernel_context(): loss.backward()
        if _grads_finite(engine):
            scaler.update(new_scale=scale0)
            AMP_EVENTS.append(dict(tag=tag,step=step,mode='fp32'))
            return float(loss.detach()),False
    raise FloatingPointError('Non-finite gradients in AMP and in fp32 alike')

def named_hash(engine):
    h=hashlib.sha256()
    for name,p in sorted(engine.params().items()):
        a=p.detach().cpu().contiguous()
        h.update(name.encode()); h.update(str(tuple(a.shape)).encode())
        h.update(a.reshape(-1).view(torch.uint8).numpy().tobytes())
    return h.hexdigest()

print('Episodic engine ready.')

## 5. Checkpointed training, and the context curve

`train_fixed` runs a number of steps **fixed in advance**, with no early stopping
and no milestone selected on a metric. It saves the weights at each milestone,
which is what makes three budgets cost only the longest training run.

`curve_for` takes one set of weights and produces a full curve along the context
axis. Small contexts are repeated and averaged, because a 32-row context is a
noisy draw. Out-of-memory at one context size skips that point and is recorded;
it never kills the run.

In [ ]:

BASE=SCRATCH/'base_weights.pth'
if not BASE.exists():
    check_disk('base weights',extra=2.0)
    e=EpisodeEngine(None,[],SEED)
    try: e.save(BASE)
    finally: e.close()
print('Official weights:',BASE,'|',round(BASE.stat().st_size/GIB,2),'Gio')

def predict_ctx(checkpoint,Xctx,yctx,Xq,seed,batch,cat_indices):
    m=TabPFNClassifier(model_path=str(checkpoint),device=DEVICE,n_estimators=1,
        fit_mode='low_memory',memory_saving_mode=True,random_state=int(seed),
        categorical_features_indices=cat_indices,ignore_pretraining_limits=True)
    try:
        m.fit(Xctx,yctx); pos=list(m.classes_).index(1); parts=[]
        for st in range(0,len(Xq),batch):
            parts.append(np.asarray(m.predict_proba(Xq[st:st+batch])[:,pos],dtype=float))
        p=np.concatenate(parts)
        assert len(p)==len(Xq) and np.isfinite(p).all()
        return p
    finally:
        del m; clean_gpu()

def ctx_indices(b,size,seed):
    # size=None => the whole training split. Otherwise a deterministic stratified subsample.
    n=len(b['ytrain']); k=n if size is None else min(int(size),n)
    return choose_indices(b['ytrain'],k,seed)

def eval_slice(b,cap,seed):
    idx=choose_indices(b['yreport'],min(cap,len(b['yreport'])),seed)
    return b['Xreport'][idx],b['yreport'][idx]

def curve_for(checkpoint,tid,arm,ft_steps,episode_rows):
    # One set of weights -> one full curve along the context axis.
    b=BUNDLES[tid]; cat=b['cat_indices']; ntr=len(b['ytrain'])
    Xq,yq=eval_slice(b,REPORT_EVAL_CAP,SEED+900+tid)
    out=[]
    for size in CONTEXT_SIZES:
        if size is not None and size>=ntr:
            continue                      # identical to the full context: do not pay for it twice
        nrep=REPEATS_SMALL if (size is not None and size<=SMALL_CONTEXT_MAX) else 1
        for r in range(nrep):
            try:
                idx=ctx_indices(b,size,SEED+tid+7919*r)
            except ValueError as exc:
                print(f'  {arm} ctx={size}: cannot draw a context ({exc})',flush=True); break
            t0=time.monotonic(); p=None
            for bs in (PRED_BATCH,16,4,1):
                try:
                    p=predict_ctx(checkpoint,b['Xtrain'][idx],b['ytrain'][idx],Xq,SEED,bs,cat)
                    break
                except RuntimeError as exc:
                    if not is_oom(exc): raise
                    clean_gpu()
            if p is None:
                print(f'  {arm} ctx={size}: OOM at {len(idx)} rows, point skipped',flush=True); break
            out.append(dict(run_hash=RUN_HASH,target=tid,name=META[tid]['name'],n_train=ntr,
                            arm=arm,ft_steps=int(ft_steps),episode_rows=int(episode_rows),
                            context_size=('full' if size is None else str(int(size))),
                            requested=(ntr if size is None else int(size)),
                            n_context_rows=len(idx),repeat=r,n_report_eval=len(yq),
                            seconds=round(time.monotonic()-t0,1),**scores(yq,p)))
    if not out: raise RuntimeError(f'{arm}: no evaluable context size')
    return out

def train_fixed(b,start,path,milestones,seed,tag,n_context,n_query):
    path=Path(path); path.mkdir(parents=True,exist_ok=True)
    done=path/'complete.json'; end=max(milestones)
    signature=sha(dict(run=RUN_HASH,start=sha_file(start),steps=sorted(milestones),
                       seed=seed,tag=tag,ep=(n_context,n_query)))
    if done.exists():
        info=json.loads(done.read_text())
        if info['signature']==signature and all(
                Path(c['path']).is_file() and sha_file(c['path'])==c['sha256']
                for c in info['checkpoints'].values()):
            print(tag,': already trained, reused.'); return info
    check_disk(tag,extra=2.0*len(milestones))
    e=EpisodeEngine(start,b['cat_indices'],seed)
    opt=torch.optim.AdamW(list(e.params().values()),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    scaler=GradScaler(); history=[]; ckpts={}; seen=set(); redraws=0
    t0=time.monotonic(); torch.cuda.reset_peak_memory_stats(); initial=named_hash(e)
    try:
        for step in range(1,end+1):
            # An episode that is non-finite even in fp32 is redrawn, never skipped:
            # the step budget stays exactly what was announced.
            for attempt in range(6):
                episode,ic,iq=ft_episode(b,seed+step*7919+attempt*104729,n_context,n_query)
                batch=e.batch(episode)
                try:
                    loss,used_amp=backward_checked(e,batch,opt,scaler,tag=tag,step=step)
                    break
                except FloatingPointError:
                    redraws+=1
                    AMP_EVENTS.append(dict(tag=tag,step=step,mode='episode_redrawn',attempt=attempt))
                    del batch; clean_gpu()
            else:
                raise FloatingPointError(
                    f'{tag}: 6 consecutive non-finite episodes at step {step}. '
                    'Reproducible instability, not a fluke: report this dataset.')
            seen.update(ic.tolist()); seen.update(iq.tolist())
            norm=float(torch.nn.utils.clip_grad_norm_(list(e.params().values()),GRAD_CLIP))
            if used_amp:
                scaler.step(opt); scaler.update()
            else:
                opt.step()                      # gradients are already at scale 1
            history.append(dict(step=step,loss=loss,grad_norm_before_clip=norm,
                                amp=used_amp,seconds=time.monotonic()-t0))
            del batch
            if step in milestones:
                ck=path/f'step_{step:04d}.pth'; e.save(ck)
                wh=named_hash(e)
                assert wh!=initial,'Weights unchanged after training'
                ckpts[str(step)]=dict(path=str(ck),sha256=sha_file(ck),weights_hash=wh,
                                      seconds=time.monotonic()-t0)
                write_csv(path/'training_history.csv',pd.DataFrame(history))
                fp32=int((~pd.DataFrame(history).amp).sum())
                print(f'  {tag} | step {step}/{end} | loss {loss:.5f} | |g| {norm:.3f}'
                      +(f' | fp32 fallbacks {fp32} | redraws {redraws}' if fp32 or redraws else ''),
                      flush=True)
        info=dict(signature=signature,initial_hash=initial,checkpoints=ckpts,
                  seconds=time.monotonic()-t0,n_train=len(b['ytrain']),
                  unique_rows_seen=len(seen),episode_rows=n_context+n_query,
                  fp32_steps=int((~pd.DataFrame(history).amp).sum()),
                  episode_redraws=redraws,
                  final_loss_mean=float(pd.DataFrame(history).loss.tail(10).mean()),
                  peak_allocated_gib=torch.cuda.max_memory_allocated()/GIB)
        write_json(done,info)
        return info
    finally:
        del opt,scaler; e.close()
        if AMP_EVENTS: write_csv(ROOT/'amp_events.csv',pd.DataFrame(AMP_EVENTS))

print('Checkpointed training and context curve ready.')

## 6. Preflight checks

Three things are verified before committing hours of compute, and nothing starts
without a PASS:

1. **memory** — which query batch fits on the T4, including at the full context
   of the largest dataset;
2. **function** — the weights move during training, reload identically, and
   produce deterministic predictions;
3. **effect** — a fine-tuned model predicts measurably differently from the base
   model, so that a null result later cannot be a plumbing failure.

In [ ]:

attempts=[]; PRED_BATCH=None
for bs in PRED_BATCHES:
    try:
        for tid in ACTIVE_IDS:
            b=BUNDLES[tid]; predict_ctx(BASE,b['Xcontext'],b['ycontext'],b['Xreport'][:bs],SEED,bs,b['cat_indices'])
        PRED_BATCH=bs; break
    except RuntimeError as exc:
        if not is_oom(exc): raise
        attempts.append(dict(batch=bs,error='CUDA OOM')); clean_gpu()
if PRED_BATCH is None: raise RuntimeError('Even a single query row does not fit in memory.')
print('Query batch size chosen:',PRED_BATCH)

big=max(ACTIVE_IDS,key=lambda t:len(BUNDLES[t]['ytrain']))
nbig=len(BUNDLES[big]['ytrain'])
try:
    predict_ctx(BASE,BUNDLES[big]['Xtrain'],BUNDLES[big]['ytrain'],
                BUNDLES[big]['Xreport'][:PRED_BATCH],SEED,PRED_BATCH,BUNDLES[big]['cat_indices'])
    print(f'Full context tested on {META[big]["name"]}: {nbig} rows, OK')
except RuntimeError as exc:
    if not is_oom(exc): raise
    clean_gpu()
    print(f'WARNING: full context of {nbig} rows ran out of memory. The ctx=full points '
          'will fall back to smaller batches, or be dropped.',flush=True)

from sklearn.datasets import load_breast_cancer
xp,yp=load_breast_cancer(return_X_y=True)
a_,v_=train_test_split(np.arange(len(yp)),train_size=400,stratify=yp,random_state=SEED)
probe=dict(Xtrain=xp[a_].astype('float32'),ytrain=yp[a_].astype(np.int64),
           Xreport=xp[v_].astype('float32'),yreport=yp[v_].astype(np.int64),
           Xcontext=xp[a_].astype('float32'),ycontext=yp[a_].astype(np.int64),cat_indices=[])
RUN_HASH='preflight'
pf=SCRATCH/'_preflight'
if pf.exists(): shutil.rmtree(pf)
try:
    info=train_fixed(probe,BASE,pf,[1,2],SEED,'preflight',FT_CONTEXT,FT_QUERY)
    ck=info['checkpoints']['2']['path']
    e=EpisodeEngine(ck,[],SEED)
    try: assert named_hash(e)==info['checkpoints']['2']['weights_hash'],'Incorrect reload'
    finally: e.close()
    q=probe['Xreport'][:16]
    p1=predict_ctx(ck,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    p2=predict_ctx(ck,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    assert np.allclose(p1,p2,atol=1e-5,rtol=1e-4),'Predictions not reproducible'
    p0=predict_ctx(BASE,probe['Xtrain'][:128],probe['ytrain'][:128],q,SEED,16,[])
    dmax=float(np.abs(p0-p1).max())
    assert dmax>1e-4,('The fine-tuned model predicts exactly like the base model: '
                      'the grid would be flat for a trivial reason.')
    print(f'PREFLIGHT PASS - weights reloaded identically, predictions deterministic, '
          f'max base/tuned gap {dmax:.4g}')
finally:
    shutil.rmtree(pf,ignore_errors=True); clean_gpu()

TASK_IDS=[t for t in ACTIVE_IDS if t in SHARD_IDS]
LONG_IDS=(sorted(TASK_IDS,key=lambda t:-len(BUNDLES[t]['ytrain']))[:LONG_PHASE_TASKS]
          if RUN_LONG_EPISODE_PHASE else [])
CONFIG=dict(protocol='context_x_finetuning_v1',seed=SEED,tasks=TASK_IDS,
    context_sizes=[str(c) for c in CONTEXT_SIZES],ref_context=REF_CONTEXT,
    repeats_small=REPEATS_SMALL,small_max=SMALL_CONTEXT_MAX,
    ft_milestones=list(FT_MILESTONES),ft_episode=(FT_CONTEXT,FT_QUERY),
    long_phase=bool(RUN_LONG_EPISODE_PHASE),long_ids=LONG_IDS,
    long_episode=(LONG_FT_CONTEXT,LONG_FT_QUERY),long_milestones=list(LONG_FT_MILESTONES),
    lr=LEARNING_RATE,wd=WEIGHT_DECAY,clip=GRAD_CLIP,
    report_cap=REPORT_EVAL_CAP,pred_batch=PRED_BATCH,
    splits='outer_80_20_then_70_15_15_v1',
    versions={k:importlib.metadata.version(k) for k in
              ['tabpfn','torch','numpy','pandas','scikit-learn','scipy']})
RUN_HASH=sha(CONFIG)[:16]
manifest=ROOT/'manifest.json'
if manifest.exists():
    assert json.loads(manifest.read_text())['run_hash']==RUN_HASH,'Different configuration: change RUN_TAG.'
else:
    write_json(manifest,dict(run_hash=RUN_HASH,config=CONFIG,memory_attempts=attempts))
RUN_T0=time.monotonic()
def hours_left():
    return TIME_BUDGET_HOURS-(time.monotonic()-RUN_T0)/3600
print('Protocol frozen:',RUN_HASH,'|',len(TASK_IDS),'datasets | phase C on',len(LONG_IDS))

## 7. Recorded prediction

Written **before** anything is looked at, and printed into the notebook output so
that it is timestamped by the run itself. This is what separates a prediction
from a retrospective explanation.

In [ ]:

PREREG = '''
H1  The gain from fine-tuning decays as the context grows.
    Test: Spearman(log2 context, log-loss gain) per dataset, at the 200-step budget,
    then a sign test over the 24 coefficients. Prediction: a majority of NEGATIVE
    coefficients, with a significant sign test.

H2  A crossing context exists.
    Test: the mean gain is > 0 at small contexts (32-128) and <= 0 at the reference
    context (1024). Prediction: the crossing lies between 128 and 512.

H3  The gain at 1024 rows reproduces the three earlier runs.
    Prediction: mean gain close to zero, slightly negative (-0.006 median AUC
    in the earlier local study). If this point does NOT reproduce, the harness has
    changed behaviour and nothing else is interpretable.

H4  (phase C) The peak of the gain moves with the length of a training episode.
    Prediction: with 640-row episodes the crossing moves to the RIGHT. If it does
    not move, the effect is down to the inference context alone.

ALTERNATIVE OUTCOME, to be written up as such if it occurs:
    gain <= 0 at EVERY context size, including 32 rows. The negative result then
    becomes stronger, since it survives the regime most favourable to fine-tuning.
    That is not a failure of the experiment, it is its result.
'''
print(PREREG)
write_json(ROOT/'preregistration.json',dict(run_hash=RUN_HASH,text=PREREG,
    written_before_any_result=True))

## 8. Phase A — the context curve of the un-adapted model

No gradients here. This is the reference for every gain reported later, and on
its own it is the first half of the story: the complete curve from 32 rows to the
full training split, on 24 datasets.

In [ ]:

CURVE_PATH=ROOT/'curves.csv'
curves=pd.read_csv(CURVE_PATH).to_dict('records') if CURVE_PATH.exists() else []
def arm_done(tid,arm):
    return any(int(r['target'])==tid and r['arm']==arm for r in curves)

print('=== PHASE A - un-adapted model ===',flush=True)
for k,tid in enumerate(TASK_IDS,1):
    if arm_done(tid,'base'):
        print(f'{k}/{len(TASK_IDS)} {META[tid]["name"]}: already done.'); continue
    if hours_left()<=0:
        print('Time budget reached: phase A stopped cleanly.',flush=True); break
    try:
        t0=time.monotonic()
        new=curve_for(BASE,tid,'base',0,0)
        curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
        c={r['context_size']:round(r['logloss'],4) for r in new if r['repeat']==0}
        print(f'{k}/{len(TASK_IDS)} {META[tid]["name"][:34]:<34} train {len(BUNDLES[tid]["ytrain"]):>5} '
              f'| {c} | {time.monotonic()-t0:.0f}s',flush=True)
    except Exception as exc:
        record_failure('base_curve',tid,META[tid]['name'],exc); clean_gpu()
print(f'\nPhase A: {len({r["target"] for r in curves if r["arm"]=="base"})} datasets, '
      f'{hours_left():.2f} h of budget left.')

## 9. Phase B — the full grid

One dataset at a time: train once up to the largest budget while saving
milestones, evaluate every milestone across the **whole** range of contexts, then
delete the weights. A single set of weights on disk at any moment, so the run
fits in the available scratch space.

In [ ]:

FT_DIR=SCRATCH/'ft'; FT_DIR.mkdir(parents=True,exist_ok=True)
TRAIN_PATH=ROOT/'training_summary.csv'
train_rows=pd.read_csv(TRAIN_PATH).to_dict('records') if TRAIN_PATH.exists() else []

def run_ft_phase(ids,milestones,n_context,n_query,prefix,label):
    global curves,train_rows
    print(f'\n=== {label} ===',flush=True)
    for k,tid in enumerate(ids,1):
        arms=[f'{prefix}_{s}' for s in milestones]
        if all(arm_done(tid,a) for a in arms):
            print(f'{k}/{len(ids)} {META[tid]["name"]}: already done.'); continue
        if hours_left()<=0:
            print('Time budget reached: phase stopped cleanly.',flush=True); break
        b=BUNDLES[tid]; path=FT_DIR/f'{prefix}_{tid}'
        print(f'\n----- {k}/{len(ids)} {META[tid]["name"]} | train {len(b["ytrain"])} '
              f'| episode {n_context+n_query} rows | {hours_left():.2f} h left -----',flush=True)
        try:
            check_disk(f'fine-tuning {tid}',extra=2.0*len(milestones))
            info=train_fixed(b,BASE,path,milestones,SEED+tid,f'{prefix} {META[tid]["name"][:24]}',
                             n_context,n_query)
            train_rows=[r for r in train_rows if not (int(r['target'])==tid and r['prefix']==prefix)]
            train_rows.append(dict(target=tid,name=META[tid]['name'],prefix=prefix,
                episode_rows=n_context+n_query,steps=max(milestones),
                seconds=info['seconds'],fp32_steps=info['fp32_steps'],
                episode_redraws=info['episode_redraws'],unique_rows_seen=info['unique_rows_seen'],
                final_loss_mean=info['final_loss_mean'],peak_gib=info['peak_allocated_gib']))
            write_csv(TRAIN_PATH,pd.DataFrame(train_rows))
            for s in milestones:
                arm=f'{prefix}_{s}'
                if arm_done(tid,arm): continue
                ck=info['checkpoints'][str(s)]['path']
                new=curve_for(ck,tid,arm,s,n_context+n_query)
                curves.extend(new); write_csv(CURVE_PATH,pd.DataFrame(curves))
                base={ (r['context_size']):r['logloss'] for r in curves
                       if int(r['target'])==tid and r['arm']=='base' and r['repeat']==0 }
                g={r['context_size']:round(base.get(r['context_size'],np.nan)-r['logloss'],4)
                   for r in new if r['repeat']==0}
                print(f'  {arm} | log-loss gain by context: {g}',flush=True)
        except Exception as exc:
            record_failure(prefix,tid,META[tid]['name'],exc); clean_gpu()
        finally:
            shutil.rmtree(path,ignore_errors=True); clean_gpu()

run_ft_phase(TASK_IDS,FT_MILESTONES,FT_CONTEXT,FT_QUERY,'ft',
             f'PHASE B - fine-tuning, episodes of {FT_CONTEXT+FT_QUERY} rows')
CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\nPhase B done. {CUR.target.nunique()} datasets, {len(CUR)} points, '
      f'{len(FAILURES)} drop(s). Budget left: {hours_left():.2f} h.')

## 10. Phase C — the length of a training episode

This answers the obvious objection to phases A and B: that the fine-tuning was
specialised for short contexts. Same datasets among the largest, same protocol,
one thing changes — training episodes of 640 rows instead of 128.

Phase C runs last and cuts itself off on the time budget. A partial result stays
readable, because the comparison is paired dataset by dataset.

In [ ]:

if RUN_LONG_EPISODE_PHASE and LONG_IDS and hours_left()>0.5:
    run_ft_phase(LONG_IDS,LONG_FT_MILESTONES,LONG_FT_CONTEXT,LONG_FT_QUERY,'ftlong',
                 f'PHASE C - episodes of {LONG_FT_CONTEXT+LONG_FT_QUERY} rows')
else:
    print('Phase C not run (disabled, or insufficient time budget).')
CUR=pd.DataFrame(curves); write_csv(CURVE_PATH,CUR)
print(f'\n{CUR.target.nunique()} datasets, {CUR.arm.nunique()} arms, {len(CUR)} measured points.')

## 11. Results

Four readings, in the order they should be made.

1. **The base curve** — context alone.
2. **The grid** — fine-tuning gain by (budget, context size).
3. **The slope** — does the gain decay as the context grows?
4. **The equivalence** — how many context doublings is the fine-tuning worth?

In [ ]:

from scipy.stats import wilcoxon, spearmanr, binomtest

CUR=pd.read_csv(CURVE_PATH)
KEY=['target','name','n_train','arm','ft_steps','episode_rows','context_size','requested']
AGG=(CUR.groupby(KEY,as_index=False)
        .agg(logloss=('logloss','mean'),auc=('auc','mean'),ap=('ap','mean'),
             n_context_rows=('n_context_rows','mean'),repeats=('repeat','count')))
AGG['ctx']=AGG.n_context_rows.astype(float)
write_csv(ROOT/'curves_agg.csv',AGG)

print('=== 1. THE BASE CURVE - context alone, un-adapted model ===')
B=AGG[AGG.arm=='base'].copy()
ref=(B[B.context_size==str(REF_CONTEXT)].set_index('target')['logloss'])
B['gain_vs_ref']=B.target.map(ref)-B.logloss
base_tab=(B.groupby('context_size')
           .agg(mean_rows=('ctx','mean'),mean_logloss=('logloss','mean'),mean_auc=('auc','mean'),
                gain_vs_1024=('gain_vs_ref','mean'),datasets=('target','nunique'))
           .sort_values('mean_rows').round(5))
display(base_tab); write_csv(ROOT/'table_base_curve.csv',base_tab.reset_index())

print('\n=== 2. THE GRID - fine-tuning gain, by budget and by context ===')
FT=AGG[AGG.arm.str.startswith('ft')].merge(
    AGG[AGG.arm=='base'][['target','context_size','logloss','auc']],
    on=['target','context_size'],suffixes=('','_base'),how='inner')
FT['gain_logloss']=FT.logloss_base-FT.logloss
FT['gain_auc']=FT.auc-FT.auc_base
FT['phase']=np.where(FT.arm.str.startswith('ftlong'),'C (long episodes)','B (short episodes)')
write_csv(ROOT/'ft_gains.csv',FT)

def grid(sub,title):
    if sub.empty: print(f'\n{title}: no data.'); return None
    t=(sub.groupby(['ft_steps','context_size'])
          .agg(mean_rows=('ctx','mean'),gain_mean=('gain_logloss','mean'),
               gain_median=('gain_logloss','median'),mean_auc_gain=('gain_auc','mean'),
               datasets_improved=('gain_logloss',lambda s:int((s>0).sum())),
               datasets=('target','nunique'))
          .reset_index().sort_values(['ft_steps','mean_rows']).round(5))
    print(f'\n{title} - positive = fine-tuning helps')
    display(t); return t

gB=grid(FT[FT.phase.str.startswith('B')],f'PHASE B - episodes of {FT_CONTEXT+FT_QUERY} rows')
gC=grid(FT[FT.phase.str.startswith('C')],f'PHASE C - episodes of {LONG_FT_CONTEXT+LONG_FT_QUERY} rows')
if gB is not None: write_csv(ROOT/'grille_phaseB.csv',gB)
if gC is not None: write_csv(ROOT/'grille_phaseC.csv',gC)

print('\nTesting the gain against zero, at every context size (phase B, largest budget)')
top=max(FT_MILESTONES); zero=[]
for cs,g in FT[(FT.phase.str.startswith('B'))&(FT.ft_steps==top)].groupby('context_size'):
    d=g.gain_logloss.dropna()
    if len(d)>=5 and d.abs().sum()>0:
        zero.append(dict(context_size=cs,mean_rows=round(float(g.ctx.mean()),1),
            gain_mean=round(float(d.mean()),5),gain_median=round(float(d.median()),5),
            wilcoxon_p=round(float(wilcoxon(d).pvalue),6),
            positive=int((d>0).sum()),n=len(d)))
ZERO=pd.DataFrame(zero).sort_values('mean_rows') if zero else pd.DataFrame()
if len(ZERO): display(ZERO); write_csv(ROOT/'test_vs_zero.csv',ZERO)

print('\n=== 3. H1 - THE SLOPE: does the gain decay with context? ===')
slopes=[]
for tid,g in FT[(FT.phase.str.startswith('B'))&(FT.ft_steps==top)].groupby('target'):
    g=g.dropna(subset=['gain_logloss'])
    if len(g)>=4 and g.ctx.nunique()>=4:
        rho,p=spearmanr(np.log2(g.ctx),g.gain_logloss)
        slopes.append(dict(target=tid,name=g.name.iloc[0],n_train=int(g.n_train.iloc[0]),
            rho=round(float(rho),4),p=round(float(p),4),points=len(g),
            gain_short=round(float(g[g.ctx<=SMALL_CONTEXT_MAX].gain_logloss.mean()),5),
            gain_long=round(float(g[g.ctx>=REF_CONTEXT].gain_logloss.mean()),5)))
SL=pd.DataFrame(slopes)
if len(SL):
    display(SL.sort_values('rho')); write_csv(ROOT/'pentes_par_dataset.csv',SL)
    neg=int((SL.rho<0).sum()); n=len(SL)
    bt=binomtest(neg,n,0.5)
    print(f'\nH1: {neg}/{n} datasets show a NEGATIVE correlation between context and gain '
          f'(sign test p={bt.pvalue:.5f}, median rho {SL.rho.median():+.3f}).')
    print('  -> prediction confirmed' if neg>n/2 and bt.pvalue<0.05 else
          '  -> prediction NOT confirmed: say so plainly.')
    d2=(SL.gain_short-SL.gain_long).dropna()
    if len(d2)>=5 and d2.abs().sum()>0:
        print(f'  gap small context - large context: {d2.mean():+.5f} on average, '
              f'{int((d2>0).sum())}/{len(d2)} positive, Wilcoxon p={wilcoxon(d2).pvalue:.5f}')
else:
    print('Not enough points per dataset to estimate a slope.')

print('\n=== 4. THE CROSSING, and how much context fine-tuning is worth ===')
cross=[]
for (tid,st),g in FT[FT.phase.str.startswith('B')].groupby(['target','ft_steps']):
    g=g.dropna(subset=['gain_logloss']).sort_values('ctx')
    if g.empty: continue
    pos=g[g.gain_logloss>0]; neg_=g[g.gain_logloss<=0]
    bb=B[B.target==tid].sort_values('ctx')
    eq=np.nan
    at_ref=g[g.context_size==str(REF_CONTEXT)]
    if len(bb)>=3 and len(at_ref):
        # how many context rows would the UN-ADAPTED model need to match the
        # fine-tuned model at the reference context?
        x=np.log2(bb.ctx.to_numpy()); y=bb.logloss.to_numpy()
        o=np.argsort(-x); yy=np.maximum.accumulate(y[o]); xx=x[o]   # increasing log-loss
        target_ll=float(at_ref.logloss.iloc[0])
        if yy.min()<=target_ll<=yy.max():
            eq=float(np.interp(target_ll,yy,xx))-np.log2(float(at_ref.ctx.iloc[0]))
    cross.append(dict(target=tid,name=g.name.iloc[0],ft_steps=int(st),
        n_train=int(g.n_train.iloc[0]),
        last_winning_context=(int(pos.ctx.max()) if len(pos) else None),
        first_losing_context=(int(neg_.ctx.min()) if len(neg_) else None),
        equivalent_doublings=(round(eq,3) if np.isfinite(eq) else None)))
CROSS=pd.DataFrame(cross)
if len(CROSS):
    display(CROSS.sort_values(['ft_steps','n_train'])); write_csv(ROOT/'croisements.csv',CROSS)
    eqs=CROSS[CROSS.ft_steps==top].equivalent_doublings.dropna()
    if len(eqs):
        print(f'\nAt a budget of {top} steps, fine-tuning is worth a median of '
              f'{eqs.median():+.2f} context doubling(s) '
              f'({int((eqs>0).sum())}/{len(eqs)} datasets where it is worth a positive one).')
        print('  Reading: a negative number means fine-tuning REMOVES '
              'the equivalent of that many context doublings.')

if len(FT[FT.phase.str.startswith('C')]):
    print('\n=== H4 - does episode length shift the curve? ===')
    cmp_=FT[FT.target.isin(LONG_IDS)&FT.ft_steps.isin(set(FT_MILESTONES)&set(LONG_FT_MILESTONES))]
    t4=(cmp_.groupby(['phase','ft_steps','context_size'])
            .agg(mean_rows=('ctx','mean'),gain_mean=('gain_logloss','mean'),
                 datasets=('target','nunique'))
            .reset_index().sort_values(['ft_steps','mean_rows','phase']).round(5))
    display(t4); write_csv(ROOT/'comparaison_episodes.csv',t4)

## 12. Figures

Three figures, and a fourth if phase C ran. Fine-tuning budgets are an **ordered**
quantity, so they are encoded as a single-hue ramp from light to dark, with
distinct markers as a second channel for readers who cannot separate the steps by
colour alone.

In [ ]:

RAMP=['#8FB4DC','#4B7FBB','#1F4E79']      # sequential ramp: increasing step budget
MARKS=['o','s','^']
ORANGE='#D1793A'; GREY='#6B7280'; BLUE='#3B6FB6'
plt.rcParams.update({'figure.dpi':160,'savefig.dpi':300,'font.size':9,
                     'axes.spines.top':False,'axes.spines.right':False,
                     'axes.edgecolor':'#9CA3AF','axes.labelcolor':'#374151',
                     'xtick.color':GREY,'ytick.color':GREY,'text.color':'#1F2937'})
def col(i): return RAMP[min(i,len(RAMP)-1)]
def mark(i): return MARKS[min(i,len(MARKS)-1)]

# --- Figure 1: the grid, the main result
def mean_by_size(sub):
    return (sub.groupby('context_size')
              .agg(x=('ctx','mean'),y=('gain_logloss','mean'),
                   sd=('gain_logloss','std'),n=('target','nunique')).sort_values('x'))

sub=FT[FT.phase.str.startswith('B')]
if len(sub):
    fig,ax=plt.subplots(figsize=(5.8,4.2))
    ax.set_xscale('log')
    ax.axhline(0,color=ORANGE,lw=2.0,zorder=2)
    for i,st in enumerate(sorted(sub.ft_steps.unique())):
        s_=sub[sub.ft_steps==st]
        g=mean_by_size(s_[s_.context_size!='full'])      # fixed sizes, comparable
        if not len(g): continue
        ax.errorbar(g.x,g.y,yerr=g.sd/np.sqrt(g.n.clip(lower=1)),color=col(i),lw=1.9,
                    marker=mark(i),ms=6,capsize=2.5,zorder=3)
        ax.annotate(f'{int(st)} steps',xy=(g.x.iloc[0],g.y.iloc[0]),xytext=(-7,0),
                    textcoords='offset points',color=col(i),fontsize=8,va='center',ha='right')
        f_=mean_by_size(s_[s_.context_size=='full'])      # x axis = mean training-set size
        if len(f_):
            ax.errorbar(f_.x,f_.y,yerr=f_.sd/np.sqrt(f_.n.clip(lower=1)),color=col(i),
                        marker='*',ms=13,ls='none',capsize=2.5,zorder=4,
                        label='full training split (averaged abscissa)' if i==0 else None)
    ax.axvline(REF_CONTEXT,color=GREY,lw=.8,ls='--',zorder=1)
    x0=0.35*float(sub.ctx.min()); x1=1.6*float(sub.ctx.max())   # room for the labels
    ax.set_xlim(x0,x1); y0,y1=ax.get_ylim()
    ax.annotate('un-adapted model',xy=(x1,0),xytext=(-4,4),textcoords='offset points',
                color=ORANGE,fontsize=8,va='bottom',ha='right')
    ax.annotate(f'earlier runs\n({REF_CONTEXT} rows)',xy=(REF_CONTEXT,y0),
                xytext=(-5,10),textcoords='offset points',color=GREY,fontsize=7.5,ha='right')
    ax.set_xlabel('Rows in the inference context (log scale)')
    ax.set_ylabel('Log-loss gain from fine-tuning')
    ax.set_title('Is fine-tuning a substitute for context?',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True)
    ax.legend(frameon=False,fontsize=7.5,loc='lower left')
    fig.tight_layout(); fig.savefig(ROOT/'fig1_grille.png',bbox_inches='tight'); plt.show()

# --- Figure 2: the base curve, in absolute terms
if len(B):
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    for tid,g in B.groupby('target'):
        g=g.sort_values('ctx'); ax.plot(g.ctx,g.logloss,lw=1.0,color=BLUE,alpha=.30,
                                        marker='o',ms=2.5,mew=0,zorder=2)
    def mean_ll(sub):
        return (sub.groupby('context_size').agg(x=('ctx','mean'),y=('logloss','mean'),
                sd=('logloss','std'),n=('target','nunique')).sort_values('x'))
    m=mean_ll(B[B.context_size!='full'])        # fixed sizes, same datasets throughout
    ax.errorbar(m.x,m.y,yerr=m.sd/np.sqrt(m.n.clip(lower=1)),color=ORANGE,lw=2.3,marker='o',
                ms=7,capsize=3,zorder=4,label='mean, fixed sizes')
    mf=mean_ll(B[B.context_size=='full'])
    if len(mf):
        ax.errorbar(mf.x,mf.y,yerr=mf.sd/np.sqrt(mf.n.clip(lower=1)),color=ORANGE,marker='*',
                    ms=15,ls='none',capsize=3,zorder=5,label='full training split (averaged abscissa)')
    ax.plot([],[],lw=1.0,color=BLUE,alpha=.5,label='one dataset')
    ax.set_xscale('log'); ax.set_xlabel('Rows in the context (log scale)')
    ax.set_ylabel('Log-loss on the sealed report split')
    ax.set_title('What context alone buys, with no training at all',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig2_courbe_base.png',bbox_inches='tight'); plt.show()

# --- Figure 3: spread across datasets at the largest budget
s3=FT[(FT.phase.str.startswith('B'))&(FT.ft_steps==max(FT_MILESTONES))]
if len(s3):
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    ax.axhline(0,color=ORANGE,lw=1.8,zorder=3)
    for tid,g in s3.groupby('target'):
        g=g.sort_values('ctx')
        ax.plot(g.ctx,g.gain_logloss,lw=1.0,color=RAMP[2],alpha=.30,marker='o',ms=2.5,
                mew=0,zorder=2)
    m3=s3.groupby('context_size').agg(x=('ctx','mean'),y=('gain_logloss','mean')).sort_values('x')
    ax.plot(m3.x,m3.y,lw=2.3,color=RAMP[2],marker='s',ms=7,zorder=4,label='moyenne')
    ax.plot([],[],lw=1.0,color=RAMP[2],alpha=.5,label='one dataset')
    ax.set_xscale('log'); ax.set_xlabel('Rows in the context (log scale)')
    ax.set_ylabel('Log-loss gain')
    ax.set_title(f'Spread across datasets, {max(FT_MILESTONES)} fine-tuning steps',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=8)
    fig.tight_layout(); fig.savefig(ROOT/'fig3_dispersion.png',bbox_inches='tight'); plt.show()

# --- Figure 4: phase C
s4=FT[FT.target.isin(LONG_IDS)] if LONG_IDS else FT.iloc[0:0]
if len(s4[s4.phase.str.startswith('C')]):
    fig,ax=plt.subplots(figsize=(5.6,4.0))
    ax.axhline(0,color=ORANGE,lw=1.8,zorder=3)
    for j,(ph,st) in enumerate(sorted({(p,s) for p,s in zip(s4.phase,s4.ft_steps)})):
        g=(s4[(s4.phase==ph)&(s4.ft_steps==st)].groupby('context_size')
             .agg(x=('ctx','mean'),y=('gain_logloss','mean')).sort_values('x'))
        if len(g)<2: continue
        short=ph.startswith('B')
        ax.plot(g.x,g.y,lw=1.9,ls='-' if short else '--',color=col(0 if short else 2),
                marker=mark(j%3),ms=6,zorder=4,
                label=f'episodes {FT_CONTEXT+FT_QUERY if short else LONG_FT_CONTEXT+LONG_FT_QUERY} rows, {int(st)} steps')
    ax.set_xscale('log'); ax.set_xlabel('Rows in the context (log scale)')
    ax.set_ylabel('Log-loss gain')
    ax.set_title('Does episode length shift the curve?',loc='left')
    ax.grid(alpha=.18,lw=.6); ax.set_axisbelow(True); ax.legend(frameon=False,fontsize=7.5)
    fig.tight_layout(); fig.savefig(ROOT/'fig4_episodes.png',bbox_inches='tight'); plt.show()

write_json(ROOT/'RUN_COMPLETE.json',dict(run_hash=RUN_HASH,
    datasets=int(CUR.target.nunique()),arms=sorted(CUR.arm.unique().tolist()),
    points=len(CUR),excluded=FAILURES,hours_left=round(hours_left(),2),
    note='development study; OpenML outer fold 0 never read'))
print('\nFiles in',ROOT)
for f in sorted(ROOT.glob('*.csv'))+sorted(ROOT.glob('*.png'))+sorted(ROOT.glob('*.json')):
    print('  ',f.name)

## How to read this run

**The first number to look at** is the mean gain at 1024 rows on the 50-step
budget, which is the exact setting of the earlier studies. If it does not land
near zero as they did, something in the harness has changed and nothing else here
is interpretable.

**Then the slope.** If the gain decays as the context grows, fine-tuning and
context are substitutes over part of the range, and the practical question
becomes which of the two is cheaper for a given budget. If the gain is flat, they
are independent and the finding is different — it must be reported as such.

**Then the sign at large contexts.** A negative gain at the full training split
is not a failure of the experiment. It says that adapting a model on short
episodes and then deploying it on a long context costs accuracy, which is a
result in its own right.

**What this run cannot say.** Nothing about multiclass or regression. Nothing
about budgets beyond 200 steps. Nothing about a mechanism. Every claim here is an
association measured on 24 binary tasks, with the sample size printed next to it.